In [1]:
import os
import json
from dotenv import load_dotenv
from groq import Groq

notebook_dir = os.getcwd()
env_path = os.path.abspath(os.path.join(notebook_dir, "..", ".env"))

if os.path.exists(env_path):
    load_dotenv(dotenv_path=env_path)
    print(f" Fichier .env chargé depuis : {env_path}")
else:
    load_dotenv()  
    print("Recherche du fichier .env dans le dossier courant.")
GROQ_API_KEY = os.getenv("GROQ_API_KEY") or os.getenv("API_KEY")
if not GROQ_API_KEY:
    GROQ_API_KEY = "gsk_fictive_key_for_testing_purposes"
    print(" Warning : Aucune clé API Groq trouvée dans votre fichier .env.")
    print("   -> Le système basculera automatiquement sur le 'Mode Secours' (alertes locales).")
else:
    print(" Clé API Groq détectée ! L'analyseur d'IA Llama-3.3 est prêt à l'emploi.")

client = Groq(api_key=GROQ_API_KEY)

 Fichier .env chargé depuis : c:\Users\Hp\Desktop\PFA_hybride\.env
 Clé API Groq détectée ! L'analyseur d'IA Llama-3.3 est prêt à l'emploi.


In [2]:
def alerte_secours(type_attaque, confiance):
    """
    Génère un dictionnaire d'alerte pré-rédigé (Mode Secours) 
    si l'accès à l'API Groq est indisponible ou non configuré.
    """
    return {
        "titre": f"Alerte Système Statique : Détection de {type_attaque}",
        "gravite": "Critique" if type_attaque in ["DoS", "DDoS", "Mirai"] else "Élevé" if type_attaque != "BenignTraffic" else "Faible",
        "description": f"Le modèle hybride (GRU+LSTM) a intercepté un comportement anormal correspondant à l'empreinte réseau de la famille d'attaque {type_attaque} avec un niveau de confiance de {confiance:.1%}.",
        "recommandations": [
            "Isoler immédiatement l'équipement IoT cible ou le segment réseau affecté.",
            "Inspecter la capture PCAP récente à l'aide d'outils d'analyse comme Wireshark ou tcpdump.",
            "Appliquer une règle d'urgence (Drop/Block) sur le pare-feu pour bloquer la source suspecte."
        ]
    }

def generer_alerte(type_attaque, confiance, details):
    """
    Envoie les données d'anomalie réseau à Llama 3.3 via Groq.
    Retourne un dictionnaire JSON contenant l'analyse d'incident et les contre-mesures.
    """
    prompt = f"""Tu es un expert en cybersécurité IoT.

Attaque détectée par notre système IDS :
- Type      : {type_attaque}
- Confiance : {confiance:.1%}
- Détails   : {json.dumps(details, ensure_ascii=False)}

Réponds uniquement en JSON valide, sans texte avant ou après (sans salutations ni commentaires) :
{{
  "titre"           : "...",
  "gravite"         : "Faible|Moyen|Élevé|Critique",
  "description"     : "...",
  "recommandations" : ["...", "...", "..."]
}}"""

    if "fictive_key" in str(client.api_key):
        return alerte_secours(type_attaque, confiance)

    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=500,
            temperature=0.3
        )

        texte = response.choices[0].message.content.strip()
        if "```json" in texte:
            texte = texte.split("```json")[1].split("```")[0]
        elif "```" in texte:
            texte = texte.split("```")[1].split("```")[0]

        return json.loads(texte.strip())

    except Exception as e:
        print(f" Erreur lors de l'appel LLM Groq ({e}). Bascule sur le mode secours.")
        return alerte_secours(type_attaque, confiance)

In [3]:
test_attaque = "Mirai"
test_confiance = 0.994  
test_details = {
    "ip_source": "192.168.1.45",
    "ip_destination": "212.85.10.4",
    "port_destination": 23,  
    "protocole": "TCP",
    "taille_paquet": 64,
    "flags": "S"            
}
print(" Envoi des données de télémétrie au LLM pour expertise...")
rapport_alerte = generer_alerte(test_attaque, test_confiance, test_details)
print(" Rapport généré avec succès.\n")

print("======================================================================")
print(f" ALERTE : {rapport_alerte.get('titre')}")
print(f"CRITICITÉ : {rapport_alerte.get('gravite')}")
print("======================================================================")
print(f" DESCRIPTION ENRICHIE :\n{rapport_alerte.get('description')}\n")
print(" REMÈDES IMMÉDIATS PROPOSÉS PAR L'IA :")
for i, reco in enumerate(rapport_alerte.get('recommandations', []), 1):
    print(f"  {i}. {reco}")
print("======================================================================")

 Envoi des données de télémétrie au LLM pour expertise...
 Rapport généré avec succès.

 ALERTE : Attaque Mirai détectée
CRITICITÉ : Élevé
 DESCRIPTION ENRICHIE :
Une attaque de type Mirai a été détectée par le système IDS, avec une confiance de 99,4%. L'attaque provient de l'adresse IP 192.168.1.45 et cible l'adresse IP 212.85.10.4 sur le port 23 en utilisant le protocole TCP.

 REMÈDES IMMÉDIATS PROPOSÉS PAR L'IA :
  1. Bloquer immédiatement l'adresse IP source 192.168.1.45 pour empêcher toute autre activité malveillante
  2. Vérifier les dispositifs IoT sur le réseau pour détecter toute infection par le malware Mirai
  3. Mettre à jour les mots de passe et les paramètres de sécurité des dispositifs IoT pour prévenir de futures attaques


In [4]:

os.makedirs("../results", exist_ok=True)
chemin_export = "../results/derniere_alerte_llm.json"
with open(chemin_export, 'w', encoding='utf-8') as f:
    json.dump(rapport_alerte, f, indent=4, ensure_ascii=False)

print(f" Fichier d'incident sauvegardé avec succès dans : {chemin_export}")

 Fichier d'incident sauvegardé avec succès dans : ../results/derniere_alerte_llm.json
